In [1]:
import pandas as pd
import plotly.graph_objects as go

lab_to_class = list(pd.read_csv("../../datasets/imagenet/classes.txt", header=None).values[0])

In [2]:
class_counts = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4H99.csv").iloc[:, 1:]
class_counts = class_counts.rename(columns={"Count Majority": "maj", "Class": "cls", "Count": "val"})
class_counts["lab"] = class_counts["cls"].map(lambda x: lab_to_class.index(x))
class_counts.head()

,cls,val,maj,Count Homogeneous,lab
0,tench,1,1,1,0
1,goldfish,1,1,1,1
2,great white shark,5,5,5,2
3,tiger shark,5,5,5,3
4,hammerhead,7,7,7,4


In [3]:
coverage = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4H99Coverage.csv").iloc[:, 1:]
coverage = coverage.rename(columns={"Coverage": "cov", "True Class": "cls", "True Label": "lab", "Proportion": "prop", "Count": "size"})
coverage = coverage.sort_values("lab")
coverage.head()

,lab,cls,count,prop,cov
98,0,tench,658,0.002715,0.4874
152,1,goldfish,562,0.002319,0.4163
549,2,great white shark,72,0.000297,0.0533
707,3,tiger shark,32,0.000132,0.0237
829,4,hammerhead,23,0.000095,0.0170


In [5]:
valleys = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4H99Valleys.csv").iloc[:, 1:]
valleys = valleys.rename(columns={"major class size": "majsize", "majority class": "cls", "major class coverage": "majcov"}).drop(columns=["idx"])
valleys["lab"] = valleys["cls"].map(lambda x: lab_to_class.index(x))
valleys = valleys.sort_values("lab")
valleys.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
5972,11736,2.384186e-07,0.009049,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
925,1680,-0.000000e+00,0.009049,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
400,688,4.339124e-05,0.009092,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
4363,8479,9.989624e-03,0.019038,minima-saddle,0.004245,1,0.693147,great white shark,1.0,1,0.0007,2
5022,9759,3.060826e-04,0.009355,minima-saddle,0.000213,1,0.693147,great white shark,1.0,1,0.0007,2


How many classes have corresponding homogeneous valleys? 

In [6]:
homo_val_owners = (class_counts["maj"] != 0).value_counts()[True]
homo_val_owners

np.int64(999)

In [7]:
homo_thresh = 0.99
valleys_homo = valleys[valleys["homogeneity"] >= homo_thresh]

How many valleys are homogeneous?

In [8]:
valleys_homo_count = len(valleys_homo)
print(f"{valleys_homo_count} / {len(valleys)}")

6227 / 6227


How many classes have at least 10% coverage in their homogeneous valleys?

In [9]:
valleys_homo.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
5972,11736,2.384186e-07,0.009049,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
925,1680,-0.000000e+00,0.009049,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
400,688,4.339124e-05,0.009092,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
4363,8479,9.989624e-03,0.019038,minima-saddle,0.004245,1,0.693147,great white shark,1.0,1,0.0007,2
5022,9759,3.060826e-04,0.009355,minima-saddle,0.000213,1,0.693147,great white shark,1.0,1,0.0007,2


In [10]:
valleys_homo["lab"].value_counts()

lab
577    44
40     39
292    34
299    33
250    32
       ..
964     1
994     1
993     1
991     1
990     1
Name: count, Length: 997, dtype: int64

In [20]:
pooled_cov = valleys_homo.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.1).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(480), np.int64(520))

In [21]:
pooled_cov = valleys_homo.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.01).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(962), np.int64(38))